# Supervised Fine-Tuning (SFT) for RQwen3

## Stage 2 of 3: Teaching the model to follow instructions

In notebook 05 we did **pre-training** — feeding raw text to the model so it learns
language from scratch. After pre-training, the model can complete text coherently,
but it can't answer questions or follow instructions. It just predicts the next word.

This notebook does **Supervised Fine-Tuning (SFT)** — Stage 2 of the pipeline:

```
[x] Stage 1: Pre-training (notebook 05)
    Random weights -> language understanding
    Data: trillions of tokens of raw web text
    Result: model that can complete text

[>] Stage 2: Supervised Fine-Tuning (THIS NOTEBOOK)
    Pre-trained model -> instruction follower
    Data: ~74k human-written Q&A conversations
    Result: model that answers questions and follows directions

[ ] Stage 3: RLHF / DPO (future notebook 07)
    Fine-tuned model -> aligned assistant
    Data: human preference rankings
    Result: model that gives helpful, high-quality answers
```

### How SFT differs from pre-training

| | Pre-training (notebook 05) | SFT (this notebook) |
|---|---|---|
| **Starting point** | Random weights | Pre-trained weights |
| **Data** | Raw text (FineWeb-Edu) | Structured conversations (Q&A pairs) |
| **Objective** | Predict next token on ALL text | Predict next token ONLY on assistant responses |
| **Learning rate** | 3e-4 (big updates) | 2e-5 (gentle adjustments) |
| **Duration** | 10,000+ steps | 3 epochs over ~74k examples |
| **What it learns** | Language, grammar, facts, reasoning | How to USE that knowledge in conversation |

### The key insight: SFT is about format, not knowledge

Pre-training is where the model learns English, facts, and reasoning.
SFT teaches the model a new **interface** — how to take a question and produce an answer.
Think of it like this:

- **Pre-training** = going to school and learning everything
- **SFT** = learning how to take an exam (same knowledge, new format)

### Our dataset mix

We're using three datasets to build a well-rounded assistant:

| Dataset | Examples | Focus |
|---------|----------|-------|
| Alpaca-Cleaned | ~52k | General instruction following |
| OpenAssistant (OASST2) | ~3-5k conversations | Multi-turn conversational depth |
| Python Code Instructions | ~18.6k | Python/CS code generation |

Combined, these cover general helpfulness, conversational ability, and CS skills.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import time
import math
import json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup

from src import CoreConfig, RQwen3
from src.utils import get_device, format_param_count

device = get_device()
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

---
## Dataset Loading & Conversion

Each dataset has a different format. We need to convert them all to a unified
**messages** format before we can train:

```python
[{"role": "user", "content": "What is Python?"}, 
 {"role": "assistant", "content": "Python is a programming language..."}]
```

### Format conversion needed

| Dataset | Raw format | Conversion |
|---------|-----------|------------|
| Alpaca | `instruction` + `input` + `output` fields | Merge instruction+input → user, output → assistant |
| OASST2 | Branching message trees with parent IDs | Extract linear conversations by following best-ranked path |
| Python Code Instructions | `instruction` + `input` + `output` (Alpaca format) | Same as Alpaca — merge instruction+input → user, output → assistant |

In [ ]:
# --- Dataset 1: Alpaca-Cleaned (52k general instructions) ---

def convert_alpaca(dataset):
    """Convert Alpaca format to unified messages."""
    converted = []
    for ex in dataset:
        # Combine instruction + optional input context
        if ex['input'] and ex['input'].strip():
            user_content = f"{ex['instruction']}\n\nInput: {ex['input']}"
        else:
            user_content = ex['instruction']

        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": ex['output']},
        ]
        converted.append({"messages": messages, "source": "alpaca"})
    return converted


alpaca_raw = load_dataset('yahma/alpaca-cleaned', split='train')
alpaca_data = convert_alpaca(alpaca_raw)
print(f'Alpaca: {len(alpaca_data):,} examples')
print(f'Sample:')
print(f'  User: {alpaca_data[0]["messages"][0]["content"][:100]}...')
print(f'  Asst: {alpaca_data[0]["messages"][1]["content"][:100]}...')

### OASST2: Extracting conversations from message trees

OASST2 is structured as **message trees**, not linear conversations. Each row is a
single message with a `parent_id` linking it to its parent. Multiple users can reply
to the same message, creating branches.

```
User: "What is Python?"              <- root (parent_id = None)
  ├── Assistant: "Python is..."       <- branch 1 (rank=0, best)
  │     └── User: "Can you give..."  
  │           └── Assistant: "Sure..."
  └── Assistant: "It's a snake..."    <- branch 2 (rank=1, worse)
```

**Our strategy**: For each tree, follow the **highest-ranked** path from root to leaf.
This gives us one high-quality linear conversation per tree.

In [ ]:
# --- Dataset 2: OpenAssistant OASST2 (multi-turn conversations) ---

def convert_oasst2(dataset):
    """Extract linear conversations from OASST2 message trees.
    
    For each tree, build parent->children map, then follow the
    highest-ranked child at each branch.
    """
    # Filter: English only, not deleted
    filtered = [ex for ex in dataset if ex['lang'] == 'en' and not ex['deleted']]
    print(f'  After filtering (English, not deleted): {len(filtered):,} messages')

    # Group by message tree
    trees = {}
    for msg in filtered:
        tree_id = msg['message_tree_id']
        if tree_id not in trees:
            trees[tree_id] = {}
        trees[tree_id][msg['message_id']] = msg

    print(f'  Unique message trees: {len(trees):,}')

    # Extract one conversation per tree (follow best-ranked path)
    converted = []
    for tree_id, messages in trees.items():
        children = {}  # parent_id -> list of child messages
        root = None
        for msg_id, msg in messages.items():
            pid = msg['parent_id']
            if pid is None:
                root = msg
            else:
                if pid not in children:
                    children[pid] = []
                children[pid].append(msg)

        if root is None:
            continue

        # Sort children by rank (lower = better, None = worst)
        for pid in children:
            children[pid].sort(
                key=lambda m: m['rank'] if m['rank'] is not None else 999
            )

        # Follow best path from root to leaf
        conversation = []
        current = root
        while current is not None:
            role = 'user' if current['role'] == 'prompter' else 'assistant'
            conversation.append({"role": role, "content": current['text']})
            kids = children.get(current['message_id'], [])
            current = kids[0] if kids else None

        # Keep conversations with at least one user + one assistant turn
        if len(conversation) >= 2 and conversation[0]['role'] == 'user':
            converted.append({"messages": conversation, "source": "oasst2"})

    return converted


print('Loading OASST2...')
oasst2_raw = load_dataset('OpenAssistant/oasst2', split='train')
oasst2_data = convert_oasst2(oasst2_raw)
print(f'OASST2: {len(oasst2_data):,} conversations')

# Show conversation length distribution
turn_counts = [len(ex['messages']) for ex in oasst2_data]
print(f'Turns per conversation: min={min(turn_counts)}, '
      f'max={max(turn_counts)}, avg={np.mean(turn_counts):.1f}')
print(f'Sample (first 2 turns):')
print(f'  User: {oasst2_data[0]["messages"][0]["content"][:100]}...')
print(f'  Asst: {oasst2_data[0]["messages"][1]["content"][:100]}...')

In [ ]:
# --- Dataset 3: Python Code Instructions (18.6k Python exercises) ---
# Uses Alpaca format, so the same convert_alpaca function works!

code_raw = load_dataset('iamtarun/python_code_instructions_18k_alpaca', split='train')
code_data = convert_alpaca(code_raw)

# Tag as code source (convert_alpaca tags as "alpaca", override)
for ex in code_data:
    ex['source'] = 'python_code'

print(f'Python Code: {len(code_data):,} examples')
print(f'Sample:')
print(f'  User: {code_data[0]["messages"][0]["content"][:100]}...')
print(f'  Asst: {code_data[0]["messages"][1]["content"][:100]}...')

In [ ]:
# --- Combine and shuffle all datasets ---

all_data = alpaca_data + oasst2_data + code_data
np.random.seed(42)
np.random.shuffle(all_data)

# Dataset composition
print('Combined dataset:')
sources = [ex['source'] for ex in all_data]
for src in ['alpaca', 'oasst2', 'python_code']:
    count = sources.count(src)
    print(f'  {src:20s}: {count:>6,} ({count/len(all_data)*100:.1f}%)')
print(f'  {"TOTAL":20s}: {len(all_data):>6,}')

---
## Chat Template & Loss Masking

### ChatML format

The Qwen3 tokenizer uses **ChatML** to structure conversations. Each message gets
wrapped in special tokens:

```
<|im_start|>user
What is Python?<|im_end|>
<|im_start|>assistant
Python is a programming language...<|im_end|>
```

### Why loss masking?

In pre-training, we computed loss on **every** token. In SFT, that would be wasteful —
we don't want the model spending capacity learning to predict what the **user** says.
We only want it to learn how to **respond**.

**Solution**: Set labels to `-100` (PyTorch's ignore index) for everything except
assistant response tokens. `F.cross_entropy(ignore_index=-100)` skips those positions.

```
Tokens:  <|im_start|> user \n What is Python? <|im_end|> \n <|im_start|> assistant \n Python is... <|im_end|>
Labels:     -100      -100 -100   -100   -100    -100    -100    -100        -100  -100  Python  is...  <|im_end|>
                                                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                                                                   ONLY these tokens get loss
```

This way, 100% of the gradient signal comes from the model's actual responses.

In [ ]:
class SFTDataset(Dataset):
    """Tokenizes conversations with ChatML template and masks non-assistant tokens."""

    def __init__(self, data, tokenizer, max_length=2048):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Cache special token IDs
        self.im_start_id = tokenizer.convert_tokens_to_ids('<|im_start|>')
        self.im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
        self.nl_id = tokenizer.encode('\n', add_special_tokens=False)[-1]
        self.ignore_index = -100

        print(f'Special tokens: im_start={self.im_start_id}, '
              f'im_end={self.im_end_id}, newline={self.nl_id}')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        messages = self.data[idx]['messages']

        # Build input_ids and labels token by token.
        # For each message:
        #   header = "<|im_start|>{role}\n"  -> always masked (label = -100)
        #   content tokens                    -> masked if user, trained if assistant
        #   footer = "<|im_end|>\n"           -> im_end trained for assistant, \n masked
        input_ids = []
        labels = []

        for msg in messages:
            role = msg['role']
            content = msg['content']

            # Header: <|im_start|>role\n
            role_ids = self.tokenizer.encode(role, add_special_tokens=False)
            header_ids = [self.im_start_id] + role_ids + [self.nl_id]
            input_ids.extend(header_ids)
            labels.extend([self.ignore_index] * len(header_ids))

            # Content
            content_ids = self.tokenizer.encode(content, add_special_tokens=False)
            input_ids.extend(content_ids)
            if role == 'assistant':
                labels.extend(content_ids)  # TRAIN on assistant tokens
            else:
                labels.extend([self.ignore_index] * len(content_ids))  # mask user tokens

            # Footer: <|im_end|>\n
            input_ids.extend([self.im_end_id, self.nl_id])
            if role == 'assistant':
                labels.extend([self.im_end_id, self.ignore_index])  # train on im_end
            else:
                labels.extend([self.ignore_index, self.ignore_index])

        # Truncate
        input_ids = input_ids[:self.max_length]
        labels = labels[:self.max_length]

        # Shift by 1: input = tokens[:-1], labels = tokens[1:]
        # (model predicts next token at each position)
        input_tensor = torch.tensor(input_ids[:-1], dtype=torch.long)
        label_tensor = torch.tensor(labels[1:], dtype=torch.long)

        return input_tensor, label_tensor

In [ ]:
def sft_collate_fn(batch):
    """Pad variable-length sequences to the longest in the batch."""
    input_ids = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    max_len = max(seq.size(0) for seq in input_ids)
    pad_id = tokenizer.pad_token_id or 0

    padded_inputs = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    padded_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)

    for i, (inp, lab) in enumerate(zip(input_ids, labels)):
        padded_inputs[i, :inp.size(0)] = inp
        padded_labels[i, :lab.size(0)] = lab

    return padded_inputs, padded_labels

In [ ]:
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-1.7B')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Create dataset (will be recreated with device-appropriate max_length later)
sft_dataset = SFTDataset(all_data, tokenizer, max_length=2048)

# --- Validate: check that loss masking works correctly ---
sample_input, sample_label = sft_dataset[0]
print(f'Sample input shape: {sample_input.shape}')
print(f'Sample label shape: {sample_label.shape}')

n_trained = (sample_label != -100).sum().item()
n_total = sample_label.size(0)
print(f'Trainable tokens: {n_trained}/{n_total} ({n_trained/n_total*100:.1f}%)')

print(f'\n--- Full conversation (decoded) ---')
print(tokenizer.decode(sample_input[:200]))

print(f'\n--- Assistant tokens only (what model learns) ---')
trained_ids = sample_label[sample_label != -100]
print(tokenizer.decode(trained_ids[:200]))

---
## Model & Optimizer

### Loading pre-trained weights

SFT starts from a **pre-trained** model — one that already understands language.
We load the checkpoint saved by notebook 05 and then fine-tune on top of it.

We only load `model_state_dict` from the checkpoint. The optimizer and scheduler
from pre-training are discarded — SFT uses its own with different settings.

### Why a lower learning rate?

Pre-training used `lr=3e-4` because the model was learning from scratch.
SFT uses `lr=2e-5` (15x lower) because:
- The weights already encode useful knowledge from pre-training
- Too-large updates would overwrite that knowledge (**catastrophic forgetting**)
- We only need small adjustments to teach the conversation format

### Note on padding

Our model only uses a causal mask, not a padding mask. Padding tokens at the end
of shorter sequences will be attended to — but since their labels are `-100`, they
produce no loss and don't affect training. This is a simplification; production
models add proper attention masking for padding.

In [ ]:
@dataclass
class SFTConfig:
    # Data
    max_seq_len: int = 2048

    # Training
    batch_size: int = 2
    grad_accum_steps: int = 8       # effective batch = batch_size * grad_accum
    num_epochs: int = 3
    learning_rate: float = 2e-5     # 15x lower than pre-training
    weight_decay: float = 0.01      # lighter than pre-training (0.1)
    warmup_ratio: float = 0.03      # 3% of total steps
    max_grad_norm: float = 1.0

    # Logging
    log_every: int = 10
    save_every_epoch: bool = True
    sample_every: int = 200

    # Paths
    checkpoint_dir: str = '../checkpoints'
    snapshot_dir: str = '../snapshots'
    pretrained_checkpoint: str = '../checkpoints/step_10000_final.pt'


sft_cfg = SFTConfig()

# --- Auto-scale for device ---
if device == 'mps':
    sft_cfg.batch_size = 1
    sft_cfg.max_seq_len = 512
    sft_cfg.grad_accum_steps = 16
    sft_cfg.num_epochs = 1           # quick test
    sft_cfg.sample_every = 50
    print('MPS detected -> Mac-friendly settings (batch=1, seq=512, 1 epoch)')
    print('For real SFT training, move to your desktop (CUDA).')
elif device == 'cuda':
    print('CUDA detected -> full SFT training settings')
else:
    sft_cfg.batch_size = 1
    sft_cfg.max_seq_len = 256
    sft_cfg.grad_accum_steps = 8
    sft_cfg.num_epochs = 1
    print('CPU detected -> minimal test settings')

# Recreate dataset with device-appropriate max_length
sft_dataset = SFTDataset(all_data, tokenizer, max_length=sft_cfg.max_seq_len)
dataloader = DataLoader(
    sft_dataset,
    batch_size=sft_cfg.batch_size,
    shuffle=True,                    # shuffle each epoch (unlike streaming)
    collate_fn=sft_collate_fn,
    drop_last=True,
)

steps_per_epoch = len(dataloader) // sft_cfg.grad_accum_steps
total_steps = steps_per_epoch * sft_cfg.num_epochs
warmup_steps = int(total_steps * sft_cfg.warmup_ratio)

print(f'\nDataset: {len(sft_dataset):,} examples')
print(f'Batches per epoch: {len(dataloader):,}')
print(f'Optimizer steps per epoch: {steps_per_epoch:,}')
print(f'Total optimizer steps: {total_steps:,}')
print(f'Warmup steps: {warmup_steps}')

In [ ]:
# Create model
model_config = CoreConfig()
model = RQwen3(model_config).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {total_params:,} params ({format_param_count(total_params)})')

# Load pre-trained weights
ckpt_path = sft_cfg.pretrained_checkpoint

# Try the configured path first, then scan for any available checkpoint
if not os.path.exists(ckpt_path):
    available = sorted([f for f in os.listdir(sft_cfg.checkpoint_dir) if f.endswith('.pt')])\
        if os.path.exists(sft_cfg.checkpoint_dir) else []
    if available:
        ckpt_path = os.path.join(sft_cfg.checkpoint_dir, available[-1])
        print(f'Configured checkpoint not found. Using latest: {ckpt_path}')

if os.path.exists(ckpt_path):
    print(f'\nLoading pre-trained weights from: {ckpt_path}')
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    pretrain_steps = ckpt.get('step', 'unknown')
    pretrain_loss = f"{ckpt['loss_history'][-1]:.4f}" if ckpt.get('loss_history') else 'unknown'
    print(f'  Pre-training steps: {pretrain_steps}')
    print(f'  Final pre-training loss: {pretrain_loss}')
    print('  Weights loaded successfully!')
else:
    print(f'\n*** WARNING: No pre-trained checkpoint found ***')
    print(f'Looked for: {sft_cfg.pretrained_checkpoint}')
    print('Proceeding with RANDOM weights. The model will learn the instruction')
    print('format but without language understanding from pre-training.')
    print('For best results, run notebook 05 first.')

In [ ]:
# --- "Before SFT" baseline: see how the model handles questions ---

test_prompts = [
    'What is a linked list?',
    'Explain the difference between a stack and a queue.',
    'Write a Python function to check if a number is prime.',
    'What is classical conditioning in psychology?',
    'How do you calculate the mean of a dataset?',
]


@torch.no_grad()
def generate_response(model, tokenizer, prompt, max_new_tokens=150):
    """Generate a response using ChatML format."""
    model.eval()
    chat_input = f'<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n'
    ids = tokenizer.encode(chat_input, return_tensors='pt').to(device)
    im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')

    for _ in range(max_new_tokens):
        if ids.size(1) >= model_config.max_seq_len:
            break
        logits = model(ids)
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() == im_end_id:
            break

    # Decode only the assistant response
    prompt_len = len(tokenizer.encode(chat_input, add_special_tokens=False))
    response_ids = ids[0, prompt_len:]
    model.train()
    return tokenizer.decode(response_ids, skip_special_tokens=True)


print('=' * 60)
print('BEFORE SFT (pre-trained, no instruction following)')
print('=' * 60)

before_responses = {}
for prompt in test_prompts:
    response = generate_response(model, tokenizer, prompt)
    before_responses[prompt] = response
    print(f'\nQ: {prompt}')
    print(f'A: {response[:200]}')

In [ ]:
# Optimizer: same decay/no-decay split as notebook 05
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if param.dim() < 2 or 'norm' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': sft_cfg.weight_decay},
    {'params': no_decay_params, 'weight_decay': 0.0},
], lr=sft_cfg.learning_rate)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Optimizer: AdamW (lr={sft_cfg.learning_rate}, wd={sft_cfg.weight_decay})')
print(f'Schedule: cosine with {warmup_steps} warmup steps -> {total_steps} total')
print(f'Decay params: {sum(p.numel() for p in decay_params):,}')
print(f'No-decay params: {sum(p.numel() for p in no_decay_params):,}')

---
## SFT Training Loop

The loop is very similar to pre-training (notebook 05), with a few key differences:

| | Pre-training | SFT |
|---|---|---|
| Data | Infinite streaming | Fixed dataset, iterate in epochs |
| Shuffle | No (streaming order) | Yes (shuffle each epoch) |
| Loss | All tokens | Only assistant tokens (`ignore_index=-100`) |
| LR | 3e-4 | 2e-5 |
| Duration | N steps | N epochs |

The magic line that makes loss masking work:
```python
loss = F.cross_entropy(logits, labels, ignore_index=-100)
```

PyTorch automatically skips positions where labels equal `-100`. No extra code needed!

### What to watch for
- **Loss should drop** in the first few hundred steps
- **Overfitting**: if loss drops then rises, the model is memorizing. Stop training.
- **Sample quality**: generated responses should become more instruction-like

In [ ]:
def save_checkpoint(model, optimizer, scheduler, step, epoch, loss_history, path):
    """Save everything needed to resume SFT training."""
    torch.save({
        'step': step,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss_history': loss_history,
        'stage': 'sft',
    }, path)
    print(f'  Checkpoint saved: {path}')


def save_snapshot(model, name, snapshot_dir):
    """Save weight stats snapshot (compatible with notebooks 03/04)."""
    snapshot = {}
    for pname, param in model.named_parameters():
        data = param.detach().cpu().float()
        snapshot[pname] = {
            'mean': data.mean().item(),
            'std': data.std().item(),
            'min': data.min().item(),
            'max': data.max().item(),
            'shape': list(data.shape),
            'histogram': np.histogram(data.numpy().flatten(), bins=50),
        }
    path = os.path.join(snapshot_dir, f'{name}.pt')
    torch.save(snapshot, path)
    return path


# Save pre-SFT snapshot for later comparison
os.makedirs(sft_cfg.snapshot_dir, exist_ok=True)
os.makedirs(sft_cfg.checkpoint_dir, exist_ok=True)
save_snapshot(model, 'pre_sft', sft_cfg.snapshot_dir)
print('Pre-SFT snapshot saved.')

In [ ]:
SAMPLE_PROMPTS = [
    'What is a linked list?',
    'Write a Python function to reverse a string.',
    'Explain what p-value means in statistics.',
]

model.train()
optimizer.zero_grad()

loss_history = []
epoch_losses = []
global_step = 0
t_start = time.time()

print(f'Starting SFT training...')
print(f'Epochs: {sft_cfg.num_epochs} | Steps/epoch: {steps_per_epoch:,}')
print(f'Effective batch: {sft_cfg.batch_size} x {sft_cfg.grad_accum_steps} = '
      f'{sft_cfg.batch_size * sft_cfg.grad_accum_steps} sequences')
print()

try:
    for epoch in range(sft_cfg.num_epochs):
        epoch_loss_accum = 0.0
        epoch_steps = 0
        accum_loss = 0.0
        micro_step = 0

        for batch_idx, (input_ids, labels) in enumerate(dataloader):
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            logits = model(input_ids)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                ignore_index=-100,   # <-- loss masking!
            )
            loss = loss / sft_cfg.grad_accum_steps
            loss.backward()
            accum_loss += loss.item()
            micro_step += 1

            # Optimizer step after accumulation
            if micro_step % sft_cfg.grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), sft_cfg.max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

                loss_history.append(accum_loss)
                epoch_loss_accum += accum_loss
                epoch_steps += 1
                global_step += 1
                accum_loss = 0.0

                # Logging
                if global_step % sft_cfg.log_every == 0:
                    elapsed = time.time() - t_start
                    lr = scheduler.get_last_lr()[0]
                    avg_loss = np.mean(loss_history[-sft_cfg.log_every:])
                    print(f'Epoch {epoch+1}/{sft_cfg.num_epochs} | '
                          f'Step {global_step}/{total_steps} | '
                          f'loss={avg_loss:.4f} | lr={lr:.2e} | '
                          f'{elapsed/60:.1f}min')

                # Sample generation
                if global_step % sft_cfg.sample_every == 0:
                    idx = (global_step // sft_cfg.sample_every) % len(SAMPLE_PROMPTS)
                    prompt = SAMPLE_PROMPTS[idx]
                    sample = generate_response(model, tokenizer, prompt)
                    print(f'  Q: {prompt}')
                    print(f'  A: {sample[:150]}')
                    model.train()

        # End of epoch
        avg_epoch_loss = epoch_loss_accum / max(epoch_steps, 1)
        epoch_losses.append(avg_epoch_loss)
        print(f'\n--- Epoch {epoch+1} complete | avg loss: {avg_epoch_loss:.4f} ---\n')

        # Save checkpoint per epoch
        if sft_cfg.save_every_epoch:
            ckpt_path = os.path.join(sft_cfg.checkpoint_dir, f'sft_epoch_{epoch+1}.pt')
            save_checkpoint(model, optimizer, scheduler, global_step, epoch+1, loss_history, ckpt_path)
            snap_path = save_snapshot(model, f'sft_epoch_{epoch+1}', sft_cfg.snapshot_dir)
            print(f'  Snapshot saved: {snap_path}')

except KeyboardInterrupt:
    print(f'\nSFT interrupted at step {global_step}.')
    ckpt_path = os.path.join(sft_cfg.checkpoint_dir, f'sft_step_{global_step}_interrupted.pt')
    save_checkpoint(model, optimizer, scheduler, global_step, epoch+1, loss_history, ckpt_path)
    print('Checkpoint saved. You can resume from this point.')

# Final save
else:
    ckpt_path = os.path.join(sft_cfg.checkpoint_dir, 'sft_final.pt')
    save_checkpoint(model, optimizer, scheduler, global_step, sft_cfg.num_epochs, loss_history, ckpt_path)
    save_snapshot(model, 'sft_final', sft_cfg.snapshot_dir)
    print(f'SFT complete. Final checkpoint: {ckpt_path}')

elapsed_total = time.time() - t_start
print(f'Total time: {elapsed_total/60:.1f} minutes')

---
## Training Progress

In [ ]:
if loss_history:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Raw + smoothed loss
    axes[0].plot(loss_history, alpha=0.3, color='steelblue', linewidth=0.5)
    window = min(50, len(loss_history) // 5 or 1)
    if window > 1:
        smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
        axes[0].plot(range(window-1, len(loss_history)), smoothed,
                     color='darkblue', linewidth=2, label=f'{window}-step avg')
    # Mark epoch boundaries
    for i in range(1, sft_cfg.num_epochs):
        ep_step = steps_per_epoch * i
        if ep_step < len(loss_history):
            axes[0].axvline(x=ep_step, color='red', linestyle='--', alpha=0.5,
                            label=f'Epoch {i+1}' if i == 1 else '')
    axes[0].set_xlabel('Optimizer Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('SFT Training Loss', fontsize=14, fontweight='bold')
    axes[0].legend()

    # Per-epoch average loss
    if epoch_losses:
        axes[1].bar(range(1, len(epoch_losses)+1), epoch_losses, color='steelblue')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Average Loss')
        axes[1].set_title('Loss Per Epoch', fontsize=14, fontweight='bold')

    # Log scale
    axes[2].plot(loss_history, alpha=0.3, color='steelblue', linewidth=0.5)
    if window > 1:
        axes[2].plot(range(window-1, len(loss_history)), smoothed,
                     color='darkblue', linewidth=2)
    axes[2].set_yscale('log')
    axes[2].set_xlabel('Optimizer Step')
    axes[2].set_ylabel('Loss (log)')
    axes[2].set_title('SFT Loss (Log Scale)', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f'Initial loss: {loss_history[0]:.4f}')
    print(f'Final loss:   {loss_history[-1]:.4f}')
    print(f'Best loss:    {min(loss_history):.4f} (step {np.argmin(loss_history)+1})')
else:
    print('No training history yet. Run the training loop first.')

---
## Evaluation: Before vs After SFT

The real test: can the model now answer questions it couldn't before?

We compare responses to the same test prompts from the "Before" baseline.
Look for:
- **Format**: Does it respond like an assistant (not just continue text)?
- **Relevance**: Is the response on-topic?
- **Quality**: Is the information correct (or at least plausible)?

In [ ]:
print('=' * 60)
print('AFTER SFT -- Side-by-Side Comparison')
print('=' * 60)

for prompt in test_prompts:
    after_response = generate_response(model, tokenizer, prompt)

    print(f'\nQ: {prompt}')
    print(f'{"─" * 55}')
    print(f'BEFORE SFT:')
    print(f'  {before_responses[prompt][:200]}')
    print(f'AFTER SFT:')
    print(f'  {after_response[:200]}')
    print()

In [ ]:
# Test responses across the three training domains

domain_prompts = {
    'General (Alpaca)': [
        'Summarize the main ideas of machine learning in 3 sentences.',
        'What are three tips for studying effectively?',
    ],
    'Conversational (OASST2)': [
        'What are the pros and cons of Python vs Java?',
        'Can you explain how neural networks learn in simple terms?',
    ],
    'Code (Python Instructions)': [
        'Write a Python function to find the factorial of a number.',
        'How do you sort a dictionary by value in Python?',
    ],
}

for domain, prompts in domain_prompts.items():
    print(f'\n{"=" * 50}')
    print(f'Domain: {domain}')
    print(f'{"=" * 50}')
    for prompt in prompts:
        response = generate_response(model, tokenizer, prompt, max_new_tokens=200)
        print(f'\nQ: {prompt}')
        print(f'A: {response[:300]}')

In [ ]:
# How much did SFT change the weights?

pre_sft_path = os.path.join(sft_cfg.snapshot_dir, 'pre_sft.pt')
post_sft_path = os.path.join(sft_cfg.snapshot_dir, 'sft_final.pt')

if os.path.exists(pre_sft_path) and os.path.exists(post_sft_path):
    pre_snap = torch.load(pre_sft_path, weights_only=False)
    post_snap = torch.load(post_sft_path, weights_only=False)

    params = list(pre_snap.keys())
    std_changes = [abs(post_snap[p]['std'] - pre_snap[p]['std']) for p in params]
    mean_changes = [abs(post_snap[p]['mean'] - pre_snap[p]['mean']) for p in params]

    # Which layers changed most?
    layer_changes = {}
    for p, sc in zip(params, std_changes):
        parts = p.split('.')
        layer_key = '.'.join(parts[:3]) if len(parts) > 3 else p
        if layer_key not in layer_changes:
            layer_changes[layer_key] = []
        layer_changes[layer_key].append(sc)

    print('Top 10 most-changed parameter groups:')
    avg_changes = {k: np.mean(v) for k, v in layer_changes.items()}
    for k, v in sorted(avg_changes.items(), key=lambda x: -x[1])[:10]:
        print(f'  {k}: avg |delta std| = {v:.6f}')

    print(f'\nOverall avg |delta std|:  {np.mean(std_changes):.6f}')
    print(f'Overall avg |delta mean|: {np.mean(mean_changes):.6f}')
    print(f'\nNote: SFT changes should be MUCH smaller than pre-training changes.')
    print('SFT gently adjusts existing knowledge, not learning from scratch.')
else:
    print('Run training first to generate snapshots for comparison.')

---
## What I Learned

### Pipeline Progress

```
[x] Stage 1: Pre-training (notebook 05)
    Random weights -> language understanding

[x] Stage 2: Supervised Fine-Tuning (THIS NOTEBOOK)
    Pre-trained model -> instruction follower

[ ] Stage 3: RLHF / DPO (future notebook 07)
    Fine-tuned model -> aligned assistant
```

### Key Concepts

**1. SFT is about format, not knowledge**
- Pre-training teaches the model language and facts
- SFT teaches the model HOW to use that knowledge in a conversation
- Same underlying knowledge, new interface

**2. Loss masking is essential**
- Without masking: model wastes capacity predicting user prompts
- With masking: 100% of gradient signal comes from assistant responses
- Implementation: `labels=-100` for non-assistant tokens, PyTorch skips them automatically

**3. Lower learning rate prevents catastrophic forgetting**
- Pre-training LR: 3e-4 (learning language from scratch)
- SFT LR: 2e-5 (gently adjusting, not rewriting)
- Too high -> model forgets pre-training
- Too low -> model doesn't learn to follow instructions

**4. Dataset diversity creates a well-rounded assistant**
- Alpaca: teaches general instruction following
- OASST2: teaches multi-turn conversation depth
- Python Code Instructions: teaches structured code generation
- Mix creates a model that can handle diverse requests

**5. Format conversion is the unglamorous but critical step**
- Real ML engineering = 80% data pipeline, 20% model
- Each dataset has its own schema
- OASST2 tree extraction was the hardest part
- Unified format enables mixing datasets freely

**6. SFT changes weights very little, but the impact is huge**
- Weight deltas from SFT << weight deltas from pre-training
- The model already "knows" language; SFT just redirects behavior
- This is why fine-tuning works with relatively little data (~74k examples)

### Next Steps
- [ ] Re-run notebooks 03/04 with SFT snapshots for visual comparison
- [ ] Run full training on desktop (3 epochs, CUDA)
- [ ] Experiment with different dataset ratios
- [ ] **Build notebook 07: RLHF/DPO** — train the model to prefer helpful responses
- [ ] Try adding custom Q&A data for personal subjects (your courses, interests)
- [ ] Implement temperature/top-p sampling for more varied generation